# zk-Crown · 00 · Environment setup and smoke test

**Task P0.3.** This notebook proves the environment works before any real work runs in it.
It mounts Drive, reports which accelerator Colab actually gave us, gets the source into the
runtime, installs dependencies, and runs a smoke test that exercises `src/utils/seeding.py`
and `src/utils/results.py` end to end.

It is a *thin wrapper*. All logic lives in `src/` (CLAUDE.md section 3). Nothing is
implemented here that a reviewer would need to open a notebook to find.

**Run order:** top to bottom. Edit only the Configuration cell.

**What it produces:** a result JSON on Drive plus a paste-ready table for the Results Ledger
(CLAUDE.md section 8.1). Colab's filesystem is wiped between sessions (section 2.1), so
every artifact is written to Drive, never to `/content`.

## 1. Configuration

In [ ]:
# ---- Edit this cell, then run the notebook top to bottom --------------------

# Where the source comes from. Two supported routes:
#   A) git clone  -> set REPO_URL to your remote.
#   B) zip unpack -> leave REPO_URL empty and set SOURCE_ZIP to a zip on Drive.
# Route B is what the GPU handoff protocol (CLAUDE.md 0.5) uses, so it must keep working.
REPO_URL = ""          # e.g. "https://github.com/BlackTensor/zkcrown.git"
BRANCH = "master"
SOURCE_ZIP = ""        # e.g. "/content/drive/MyDrive/zk-crown/handoff/P0.5_colab.zip"

# Drive is the only durable storage. Checkpoints and results go here.
DRIVE_ROOT = "/content/drive/MyDrive/zk-crown"

# Ephemeral working copy of the code inside the runtime.
WORK_DIR = "/content/zk-crown"

SEED = 1337

## 2. Mount Drive

Colab wipes `/content` between sessions and after a ~90 minute idle timeout. Drive is
mounted first so nothing downstream can accidentally write somewhere that evaporates.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    # Lets the notebook be smoke-tested off-Colab. DRIVE_ROOT becomes a local dir.
    DRIVE_ROOT = str(Path.cwd() / "_local_drive")
    print("Not running in Colab. DRIVE_ROOT ->", DRIVE_ROOT)

for sub in ("results", "checkpoints", "figures", "handoff"):
    Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT =", DRIVE_ROOT)
print(sorted(p.name for p in Path(DRIVE_ROOT).iterdir()))

## 3. What accelerator did we actually get?

CLAUDE.md 2.1: a T4 is the *expected* allocation, not a guaranteed one, and at peak times
there may be no GPU at all. Record what this session got rather than assuming.

In [ ]:
!nvidia-smi || echo "nvidia-smi unavailable -- no GPU attached to this runtime"

In [ ]:
import platform
import shutil
import subprocess

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "accelerator": None,
    "vram_gb": None,
    "ram_gb": None,
    "disk_free_gb": None,
    "torch": None,
    "cuda": None,
}

# System RAM. /proc/meminfo on Linux; psutil elsewhere if present.
try:
    meminfo = Path("/proc/meminfo").read_text()
    total_kb = int(next(l for l in meminfo.splitlines() if l.startswith("MemTotal")).split()[1])
    ENV["ram_gb"] = round(total_kb / 1024**2, 2)
except (OSError, StopIteration):
    try:
        import psutil

        ENV["ram_gb"] = round(psutil.virtual_memory().total / 1024**3, 2)
    except ImportError:
        pass

ENV["disk_free_gb"] = round(shutil.disk_usage("/content" if IN_COLAB else ".").free / 1024**3, 2)

try:
    import torch

    ENV["torch"] = torch.__version__
    if torch.cuda.is_available():
        ENV["accelerator"] = torch.cuda.get_device_name(0)
        ENV["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        ENV["cuda"] = torch.version.cuda
    else:
        ENV["accelerator"] = "cpu"
except ImportError:
    print("torch not importable yet; section 5 installs dependencies.")

for k, v in ENV.items():
    print(f"{k:>14} : {v}")

if ENV["accelerator"] in (None, "cpu"):
    print("\nNOTE: no GPU in this runtime. Runtime > Change runtime type > T4 GPU.")
    print("CPU-only work (crypto, Circom, auditor) is fine here; training is not.")

## 4. Get the source into the runtime

Route A clones the remote. Route B unpacks a handoff zip from Drive, which is what
CLAUDE.md 0.5 prescribes for `[GPU]` tasks. Exactly one of them must be configured.

In [ ]:
import zipfile

work = Path(WORK_DIR)
SOURCE_MODE = None  # recorded in the result JSON, so it has to reflect reality

if REPO_URL:
    SOURCE_MODE = "clone"
    if work.exists():
        print("Working copy exists; fetching instead of re-cloning.")
        subprocess.run(["git", "-C", str(work), "fetch", "--depth", "1", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(work), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(work), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(work)], check=True
        )
    head = subprocess.run(
        ["git", "-C", str(work), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    )
    print("cloned at commit", head.stdout.strip())

elif SOURCE_ZIP:
    SOURCE_MODE = "zip"
    zip_path = Path(SOURCE_ZIP)
    if not zip_path.is_file():
        raise FileNotFoundError(f"SOURCE_ZIP not found on Drive: {zip_path}")
    work.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(work)
    print(f"unpacked {zip_path.name} -> {work}")

elif Path.cwd().joinpath("src", "utils", "results.py").is_file():
    # Running from an existing checkout (e.g. locally). Use it as-is.
    SOURCE_MODE = "cwd"
    work = Path.cwd()
    print("using the current directory as the working copy:", work)

else:
    raise RuntimeError(
        "No source configured. Set REPO_URL (route A) or SOURCE_ZIP (route B) "
        "in the Configuration cell."
    )

# A zip may contain a single top-level folder; descend to wherever src/ actually is.
if not (work / "src").is_dir():
    candidates = [d for d in work.iterdir() if d.is_dir() and (d / "src").is_dir()]
    if not candidates:
        raise RuntimeError(f"no src/ directory found under {work}")
    work = candidates[0]
    print("descended into", work)

REPO_DIR = str(work)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("REPO_DIR =", REPO_DIR)
print(sorted(p.name for p in Path(REPO_DIR).iterdir() if not p.name.startswith(".")))

## 5. Dependencies

`torch` and `torchvision` are **skipped on purpose**. Colab ships builds matched to its CUDA
driver; pip-installing over them can pull a wheel that does not match, and the failure shows
up later as a CUDA error inside a training run rather than here.

In [ ]:
PREINSTALLED = {"torch", "torchvision"}

req_path = Path(REPO_DIR, "requirements.txt")
wanted = []
for line in req_path.read_text(encoding="utf-8").splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    name = line.split("==")[0].split(">=")[0].split("[")[0].strip().lower()
    if name in PREINSTALLED:
        print(f"skipping {name} (Colab ships a CUDA-matched build)")
        continue
    wanted.append(line)

print("installing:", wanted)
if wanted:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)
print("done")

In [ ]:
# Record the versions that actually landed. CLAUDE.md 2.4: pin these in
# requirements.txt the first time the full pipeline works, and copy them into
# section 8.1. ezkl in particular is beta software whose API moves between releases.
import importlib.metadata as imd

for pkg in ("torch", "torchvision", "numpy", "onnx", "onnxruntime", "ezkl", "cryptography", "streamlit"):
    try:
        print(f"{pkg:>14} == {imd.version(pkg)}")
    except imd.PackageNotFoundError:
        print(f"{pkg:>14} == (not installed)")

## 6. Smoke test

Three things, in order of how annoying they are to discover later:

1. `set_seed` is importable and actually reproducible.
2. A tensor round-trips through the GPU (skipped cleanly if there is no GPU).
3. `write_result` produces a valid record **on Drive**, so results survive the session.

In [ ]:
import random

from src.utils.results import read_result, write_result
from src.utils.seeding import set_seed

backends = set_seed(SEED)
first = [random.random() for _ in range(5)]
set_seed(SEED)
second = [random.random() for _ in range(5)]

assert first == second, "set_seed is not reproducible -- stop and fix this before anything else"
print("seeding reproducible:", backends)

In [ ]:
gpu_check = {"ran": False}

try:
    import torch

    if torch.cuda.is_available():
        set_seed(SEED)
        a = torch.randn(512, 512, device="cuda")
        out = (a @ a.T).sum().item()
        gpu_check = {
            "ran": True,
            "device": torch.cuda.get_device_name(0),
            "matmul_checksum": out,
            "peak_mem_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 2),
        }
        print("GPU matmul ok:", gpu_check)
    else:
        print("no CUDA device; skipping the GPU round-trip")
except ImportError:
    print("torch not installed; skipping the GPU round-trip")

In [ ]:
# Results go to Drive, not to the ephemeral repo checkout.
record_path = write_result(
    name="p0.3_environment_smoke",
    seed=SEED,
    task="P0.3",
    params={"repo_dir": REPO_DIR, "drive_root": DRIVE_ROOT, "source": SOURCE_MODE},
    metrics={"seeding_reproducible": True, "gpu_check": gpu_check},
    seeded_backends=backends,
    notes="Environment smoke test only. No model was trained and nothing was measured "
    "about watermarking.",
    out_dir=Path(DRIVE_ROOT, "results"),
)

print("wrote", record_path)
print()
print(record_path.read_text(encoding="utf-8"))

assert read_result(record_path)["metrics"]["seeding_reproducible"] is True
print("\nSMOKE TEST PASSED")

## 7. Numbers for the Results Ledger

Copy the rows below into CLAUDE.md section 8.1, replacing the `TBD`s. These are measured
from this session, which is the only reason they are allowed in the repo at all
(CLAUDE.md 0.3).

In [ ]:
import importlib.metadata as imd


def _or_tbd(value):
    return "TBD" if value in (None, "") else value


rows = [
    ("Accelerator assigned", _or_tbd(ENV["accelerator"])),
    ("System RAM available", f"{ENV['ram_gb']} GB" if ENV["ram_gb"] else "TBD"),
    ("torch version", _or_tbd(ENV["torch"])),
]
try:
    rows.append(("ezkl version (pinned)", imd.version("ezkl")))
except imd.PackageNotFoundError:
    rows.append(("ezkl version (pinned)", "TBD"))
rows.append(("circom / snarkjs / circomlib versions", "TBD (installed in P7.2)"))

print("| Item | Value |")
print("|---|---|")
for item, value in rows:
    print(f"| {item} | {value} |")

if ENV["vram_gb"]:
    print(f"\n(VRAM on this device: {ENV['vram_gb']} GB)")

## 8. Bring back

- `{DRIVE_ROOT}/results/p0.3_environment_smoke__*.json` — commit this to `results/`.
- The section 8.1 table printed above.

If the smoke test failed, do **not** tick P0.3. Report the traceback instead; per
CLAUDE.md 0.1 the checkbox becomes `[!]` with a note on the blocker.